<a href="https://colab.research.google.com/github/sushants05-ops/ECGR-4105---Intro-to-Machine-Learning/blob/main/ECGR_4105_Homework_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import time
import copy
import random
import csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from IPython.display import display, Image
import glob
import os

for file in sorted(glob.glob("Homework7_Results/*.png")):
    print(os.path.basename(file))
    display(Image(filename=file))

# Problem 1 and 2: CIFAR-10 Fully Connected Neural Networks

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Training configuration
BATCH_SIZE = 64
MAX_EPOCHS = 10
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0

# Network configuration
BASE_WIDTH = 512
DEEP_WIDTH = 512
NUM_CLASSES = 10

# Output folder
OUTPUT_DIR = "Homework7_Results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# CIFAR-10 preprocessing
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.4914, 0.4822, 0.4465),
        std=(0.2470, 0.2435, 0.2616)
    ),
    transforms.Lambda(lambda x: x.view(-1))
])

# Download and load CIFAR-10
train_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

# CIFAR-10 has 32 x 32 RGB images
INPUT_SIZE = 3 * 32 * 32

# Define a fully connected neural network
class FullyConnectedNN(nn.Module):

    def __init__(self, hidden_sizes, activation="relu"):
        super(FullyConnectedNN, self).__init__()

        layers = []
        input_size = INPUT_SIZE

        activation_functions = {
            "relu": nn.ReLU,
            "sigmoid": nn.Sigmoid,
            "tanh": nn.Tanh
        }

        activation_function = activation_functions[activation]

        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(input_size, hidden_size))
            layers.append(activation_function())
            input_size = hidden_size

        layers.append(nn.Linear(input_size, NUM_CLASSES))

        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)


# Count trainable parameters
def count_parameters(model):
    return sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )


# Evaluate the model
def evaluate_model(model, loader, criterion):
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * labels.size(0)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    average_loss = total_loss / total
    accuracy = 100.0 * correct / total

    return average_loss, accuracy


# Train one neural network
def train_model(model, train_loader, test_loader, experiment_name):

    model = model.to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    train_losses = []
    train_accuracies = []
    test_losses = []
    test_accuracies = []

    best_accuracy = 0.0
    best_epoch = 0
    best_model_weights = copy.deepcopy(model.state_dict())

    start_time = time.time()

    for epoch in range(MAX_EPOCHS):

        model.train()

        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(outputs, labels)

            loss.backward()

            optimizer.step()

            running_loss += loss.item() * labels.size(0)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        epoch_loss = running_loss / total
        epoch_accuracy = 100.0 * correct / total

        test_loss, test_accuracy = evaluate_model(
            model,
            test_loader,
            criterion
        )

        train_losses.append(epoch_loss)
        train_accuracies.append(epoch_accuracy)
        test_losses.append(test_loss)
        test_accuracies.append(test_accuracy)

        if test_accuracy > best_accuracy:
            best_accuracy = test_accuracy
            best_epoch = epoch + 1
            best_model_weights = copy.deepcopy(model.state_dict())

        print(
            f"{experiment_name} | "
            f"Epoch {epoch + 1}/{MAX_EPOCHS} | "
            f"Training Loss: {epoch_loss:.4f} | "
            f"Training Accuracy: {epoch_accuracy:.2f}% | "
            f"Evaluation Accuracy: {test_accuracy:.2f}%"
        )

    training_time = time.time() - start_time

    # Restore the best evaluation checkpoint
    model.load_state_dict(best_model_weights)

    final_train_loss, final_train_accuracy = evaluate_model(
        model,
        train_loader,
        criterion
    )

    final_test_loss, final_test_accuracy = evaluate_model(
        model,
        test_loader,
        criterion
    )

    total_parameters = count_parameters(model)

    print("\nExperiment:", experiment_name)
    print("Training time:", round(training_time, 2), "seconds")
    print("Selected epoch:", best_epoch)
    print("Training loss:", round(final_train_loss, 4))
    print("Training accuracy:", round(final_train_accuracy, 2), "%")
    print("Evaluation accuracy:", round(final_test_accuracy, 2), "%")
    print("Trainable parameters:", total_parameters)

    # Save the trained model
    torch.save(
        model.state_dict(),
        os.path.join(OUTPUT_DIR, experiment_name + ".pth")
    )

    # Save training curves
    plt.figure(figsize=(9, 5))
    plt.plot(range(1, len(train_losses) + 1), train_losses,
             label="Training Loss")
    plt.plot(range(1, len(test_losses) + 1), test_losses,
             label="Evaluation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Cross Entropy Loss")
    plt.title(experiment_name + " Loss")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(
        os.path.join(OUTPUT_DIR, experiment_name + "_loss.png")
    )
    plt.close()

    plt.figure(figsize=(9, 5))
    plt.plot(range(1, len(train_accuracies) + 1), train_accuracies,
             label="Training Accuracy")
    plt.plot(range(1, len(test_accuracies) + 1), test_accuracies,
             label="Evaluation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy (%)")
    plt.title(experiment_name + " Accuracy")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(
        os.path.join(OUTPUT_DIR, experiment_name + "_accuracy.png")
    )
    plt.close()

    return {
        "Experiment": experiment_name,
        "Epochs": best_epoch,
        "Training Time (seconds)": round(training_time, 2),
        "Training Loss": round(final_train_loss, 4),
        "Training Accuracy (%)": round(final_train_accuracy, 2),
        "Evaluation Accuracy (%)": round(final_test_accuracy, 2),
        "Trainable Parameters": total_parameters
    }


# Run all six experiments
results = []

experiments = [
    {
        "name": "Problem_1a_Baseline_ReLU",
        "hidden_sizes": [BASE_WIDTH, BASE_WIDTH],
        "activation": "relu"
    },
    {
        "name": "Problem_1b_Four_Layers_ReLU",
        "hidden_sizes": [DEEP_WIDTH] * 4,
        "activation": "relu"
    },
    {
        "name": "Problem_2a_Ten_Layers_ReLU",
        "hidden_sizes": [DEEP_WIDTH] * 10,
        "activation": "relu"
    },
    {
        "name": "Problem_2b_Experiment1_Wide_ReLU",
        "hidden_sizes": [BASE_WIDTH * 2, BASE_WIDTH * 2],
        "activation": "relu"
    },
    {
        "name": "Problem_2b_Experiment2_Sigmoid",
        "hidden_sizes": [BASE_WIDTH, BASE_WIDTH],
        "activation": "sigmoid"
    },
    {
        "name": "Problem_2b_Experiment3_Tanh",
        "hidden_sizes": [BASE_WIDTH, BASE_WIDTH],
        "activation": "tanh"
    }
]

for experiment in experiments:

    print("\nStarting:", experiment["name"])

    model = FullyConnectedNN(
        hidden_sizes=experiment["hidden_sizes"],
        activation=experiment["activation"]
    )

    print("Model architecture:")
    print(model)

    result = train_model(
        model,
        train_loader,
        test_loader,
        experiment["name"]
    )

    results.append(result)

    # Save updated results after every experiment
    results_df = pd.DataFrame(results)
    results_df.to_csv(
        os.path.join(OUTPUT_DIR, "Homework7_Comparison.csv"),
        index=False
    )


# Display final comparison table
results_df = pd.DataFrame(results)

print("\nFinal Homework 7 Results")
print(results_df.to_string(index=False))

# Save final results
results_df.to_csv(
    os.path.join(OUTPUT_DIR, "Homework7_Final_Results.csv"),
    index=False
)

# Generate model size comparison
plt.figure(figsize=(10, 6))
plt.bar(
    results_df["Experiment"],
    results_df["Trainable Parameters"]
)
plt.xlabel("Experiment")
plt.ylabel("Trainable Parameters")
plt.title("Model Size Comparison")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.savefig(
    os.path.join(OUTPUT_DIR, "Model_Size_Comparison.png")
)
plt.close()

# Generate training time comparison
plt.figure(figsize=(10, 6))
plt.bar(
    results_df["Experiment"],
    results_df["Training Time (seconds)"]
)
plt.xlabel("Experiment")
plt.ylabel("Training Time (seconds)")
plt.title("Training Time Comparison")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.savefig(
    os.path.join(OUTPUT_DIR, "Training_Time_Comparison.png")
)
plt.close()

# Generate evaluation accuracy comparison
plt.figure(figsize=(10, 6))
plt.bar(
    results_df["Experiment"],
    results_df["Evaluation Accuracy (%)"]
)
plt.xlabel("Experiment")
plt.ylabel("Evaluation Accuracy (%)")
plt.title("Evaluation Accuracy Comparison")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.savefig(
    os.path.join(OUTPUT_DIR, "Evaluation_Accuracy_Comparison.png")
)
plt.close()

print("\nAll experiments completed successfully.")
print("Results and plots are saved in:", OUTPUT_DIR)

Using device: cpu


100%|██████████| 170M/170M [37:40<00:00, 75.4kB/s]



Starting: Problem_1a_Baseline_ReLU
Model architecture:
FullyConnectedNN(
  (network): Sequential(
    (0): Linear(in_features=3072, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)
Problem_1a_Baseline_ReLU | Epoch 1/10 | Training Loss: 1.6623 | Training Accuracy: 41.17% | Evaluation Accuracy: 45.87%
Problem_1a_Baseline_ReLU | Epoch 2/10 | Training Loss: 1.4786 | Training Accuracy: 47.71% | Evaluation Accuracy: 47.03%
Problem_1a_Baseline_ReLU | Epoch 3/10 | Training Loss: 1.3927 | Training Accuracy: 50.71% | Evaluation Accuracy: 50.16%
Problem_1a_Baseline_ReLU | Epoch 4/10 | Training Loss: 1.3193 | Training Accuracy: 53.48% | Evaluation Accuracy: 49.47%
Problem_1a_Baseline_ReLU | Epoch 5/10 | Training Loss: 1.2656 | Training Accuracy: 55.32% | Evaluation Accuracy: 50.66%
Problem_1a_Baseline_ReLU | Epoch 6/10 | Training Loss: 1.2157 | Training Accu